<a href="https://colab.research.google.com/github/kedwinparga21-wq/Proyecto-integrado/blob/main/ea2_exploracion_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EA2 – Ideación, diseño y documentación de la necesidad
**Proyecto integrador – Analítica de Datos** · Etapa 1: obtención y descripción básica del dataset

Flujo de este notebook: **obtener el dataset → describirlo brevemente → generar su resumen en `.txt`**.

**Necesidad (escenario hipotético):** la gerencia comercial de una cadena minorista necesita saber dónde se concentran las ventas (categorías, tiendas, periodos y segmentos de clientes) para orientar sus decisiones.

In [ ]:
# Importar las dos clases desde sus archivos .py.
# Si el notebook corre en Colab y los archivos no están, se descargan del repositorio.
import urllib.request
from pathlib import Path

BASE_REPO = "https://raw.githubusercontent.com/kedwinparga21-wq/Proyecto-integrado/main/"
for archivo in ("hf_dataset_loader.py", "dataset_summarizer.py"):
    if not Path(archivo).exists():
        urllib.request.urlretrieve(BASE_REPO + archivo, archivo)

import pandas as pd
from hf_dataset_loader import HFDatasetLoader
from dataset_summarizer import DatasetSummarizer

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
print("Clases importadas: HFDatasetLoader y DatasetSummarizer")

Clases importadas: HFDatasetLoader y DatasetSummarizer


## 1. Obtención de datos
Se obtiene el dataset de Hugging Face mediante código y se carga en un DataFrame.

In [ ]:
cargador = HFDatasetLoader("infinite-dataset-hub/RetailSalesPerformance", split="train")
df = cargador.load()
print("Origen:", cargador.describe_source())

README.md:   0%|          | 0.00/1.75k [00:00<?, ?B/s]

data.csv:   0%|          | 0.00/7.80k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100 [00:00<?, ? examples/s]

Origen: {'dataset_id': 'infinite-dataset-hub/RetailSalesPerformance', 'url': 'https://huggingface.co/datasets/infinite-dataset-hub/RetailSalesPerformance', 'split': 'train', 'config_name': None, 'revision': None}


## 2. Descripción básica en Jupyter

**Algunos registros**

In [ ]:
df.head()

,idx,Date,OutletName,Location,ProductCategory,UnitsSold,Revenue,CustomerAgeGroup,CustomerIncomeLevel,Label
0,0,2023-01-01,Green Market,Springfield,Electronics,150,45000,30-45,High,High
1,1,2023-01-02,City Mall,Springfield,Home & Garden,200,50000,25-35,Medium,Medium
2,2,2023-01-03,Book Haven,Lakeview,Books,120,18000,18-24,Low,Low
3,3,2023-01-04,FashionFiesta,River City,Apparel,250,75000,40-50,High,High
4,4,2023-01-05,Food World,Springfield,Food & Beverages,300,60000,20-30,Medium,Medium


**Dimensiones** (filas y columnas)

In [ ]:
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

Filas: 100
Columnas: 10


**Nombres y tipos de las columnas**

In [ ]:
df.dtypes

,0
idx,int64
Date,object
OutletName,object
Location,object
ProductCategory,object
UnitsSold,int64
Revenue,int64
CustomerAgeGroup,object
CustomerIncomeLevel,object
Label,object


**Valores nulos por columna**

In [ ]:
df.isna().sum()

,0
idx,0
Date,0
OutletName,0
Location,0
ProductCategory,0
UnitsSold,0
Revenue,0
CustomerAgeGroup,0
CustomerIncomeLevel,0
Label,0


**Rango de fechas y variedad de las variables clave**

In [ ]:
fechas = pd.to_datetime(df["Date"], errors="coerce")
print("Periodo:", fechas.min().date(), "a", fechas.max().date())
for columna in ["OutletName", "Location", "ProductCategory", "CustomerAgeGroup", "CustomerIncomeLevel", "Label"]:
    print(f"{columna}: {df[columna].nunique()} valores distintos")

Periodo: 2023-01-01 a 2023-02-23
OutletName: 98 valores distintos
Location: 55 valores distintos
ProductCategory: 41 valores distintos
CustomerAgeGroup: 29 valores distintos
CustomerIncomeLevel: 3 valores distintos
Label: 3 valores distintos


## 3. ¿Qué representa cada fila y qué variables respaldan la necesidad?
- **Cada fila** es un registro de ventas: una tienda (`OutletName`, en una `Location`) vendió `UnitsSold` unidades de una `ProductCategory` en una `Date`, con unos `Revenue`, y se registra el perfil del cliente (`CustomerAgeGroup` y `CustomerIncomeLevel`).
- **Variables que respaldan la necesidad:** `ProductCategory`, `OutletName`, `Location`, `Date`, `UnitsSold`, `Revenue`, `CustomerAgeGroup` y `CustomerIncomeLevel` permiten comparar ventas por producto, lugar, tiempo y segmento de cliente.
- `Label` es una etiqueta de desempeño (High, Medium, Low) según la ficha del dataset.
- **Limitación:** el dataset no tiene costos, así que no permite hablar de rentabilidad, y es sintético (generado con IA).

## 4. Generación automática del resumen `.txt`
La clase `DatasetSummarizer` genera el archivo a partir del mismo DataFrame revisado en este notebook.

In [ ]:
resumidor = DatasetSummarizer(
    df,
    titulo="Resumen del DataFrame - EA2",
    origen=cargador.describe_source(),
    notas=["DataFrame completo, sin muestreo (split train).", "Dataset sintético según su ficha de Hugging Face."],
)
ruta_txt = resumidor.guardar("AE2/resultados/resumen_dataframe.txt")
print("Archivo generado en:", ruta_txt)
print()
print(ruta_txt.read_text(encoding="utf-8"))

Archivo generado en: AE2/resultados/resumen_dataframe.txt

RESUMEN DEL DATAFRAME - EA2
Generado: 2026-11-26 00:58:00
dataset_id: infinite-dataset-hub/RetailSalesPerformance
url: https://huggingface.co/datasets/infinite-dataset-hub/RetailSalesPerformance
split: train
config_name: None
revision: None

Filas: 100
Columnas: 10

Columnas, tipos de datos y valores nulos:
                       tipo  nulos
idx                   int64      0
Date                 object      0
OutletName           object      0
Location             object      0
ProductCategory      object      0
UnitsSold             int64      0
Revenue               int64      0
CustomerAgeGroup     object      0
CustomerIncomeLevel  object      0
Label                object      0
Total de valores nulos: 0

Estadísticas descriptivas (variables numéricas):
           count      mean       std     min       25%      50%       75%       max
idx        100.0     49.50     29.01     0.0     24.75     49.5     74.25      99.0
Uni

In [ ]:
# Solo en Google Colab: descarga el resumen para subirlo al repositorio
try:
    from google.colab import files
    files.download(str(ruta_txt))
except ImportError:
    print("Fuera de Colab: el archivo está en", ruta_txt)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>